In [1]:
import scipy.io
import scipy.io.wavfile
import numpy as np
import os
#import scipy.io.wavfile as wavfile
#from collections import defaultdict
# from IPython.display import Audio
# from pydub.effects import normalize

# from pydub import AudioSegment
# from pydub.playback import play 

In [17]:
class Impulse(object):
    
    def __init__(self, file=None, fs=44100.):
        self.modes = []
        self.signal = []
        self.fs = fs
        
        if file is not None:
            self.material = file.split('/')[1].split('_')[0]
            self.id = file.strip('.mat').split('_')[1]
            mat = scipy.io.loadmat(file)
            M = mat['modes'].shape[1]
            for m in range(M):
                cf    = mat['modes'][:,m][0][0][0][0]
                amp   = mat['modes'][:,m][0][1][0][0]
                decay = mat['modes'][:,m][0][2][0][0]
                self.modes.append((cf, amp, decay))
            self.modes.sort()
        else:
            self.material = 'Unknown'
            self.id = 'NA'

    # 
    def compile_signal(self, Npts=11048, fs=44100., k=0.5, m=40., decayDB=40):
        self.fs = fs
        self.signal = np.zeros(Npts)
        tt = np.arange(0, Npts) / fs
        force = np.sin(np.sqrt(k * m) * tt[tt < np.pi * m / k])
        
        for cf, OnPwr, Dcy in self.modes:
            tt = np.arange(0, Npts) / fs
            md = np.cos(2 * np.pi * cf * tt)
            md *= 10 ** (OnPwr / 20)  # Scale by the onset power
            md *= 10 ** (-tt * (decayDB / Dcy) / 20)  # Impose decay
            self.signal += md
        
        self.signal = np.convolve(force, self.signal, mode='same')
        self.signal /= np.max(np.abs(self.signal))  # Normalize the signal
        return self.signal.astype(np.float32)


In [18]:
# Load the sample parameters from Traer et al., (2019)

samples = []
for file in os.listdir('tmp/'):
    if 'mat' in file:
        samples += [Impulse('tmp/'+file)]


In [19]:
# Mean parameters for each material

mat_modes_cf  = {s.material : np.zeros(15) for s in samples}
mat_modes_amp = {s.material : np.zeros(15) for s in samples}
mat_modes_dec = {s.material : np.zeros(15) for s in samples}
for s in samples:
    for m in range(len(s.modes)):
        mat_modes_cf[s.material][m]  += s.modes[m][0]
        mat_modes_amp[s.material][m] += s.modes[m][1]
        mat_modes_dec[s.material][m] += s.modes[m][2]
for k in mat_modes_cf.keys():
    mat_modes_cf[k]  = mat_modes_cf[k]  / 5.0
    mat_modes_amp[k] = mat_modes_amp[k] / 5.0
    mat_modes_dec[k] = mat_modes_dec[k] / 5.0

In [20]:

def make_continuum(mat1, mat2, N=100):

    delta_cf  = mat_modes_cf[mat1] - mat_modes_cf[mat2]
    delta_amp = mat_modes_amp[mat1] - mat_modes_amp[mat2]
    delta_dec = mat_modes_dec[mat1] - mat_modes_dec[mat2]

    # Steps
    continuum = []
    for i in np.linspace(0, 1, 100):
        new = Impulse()
        new_cf  = mat_modes_cf[mat2]  + i * delta_cf
        new_amp = mat_modes_amp[mat2] + i * delta_amp
        new_dec = mat_modes_dec[mat2] + i * delta_dec
        new.modes = [(cf, a, b) for cf, a, b in zip(new_cf, new_amp, new_dec)]
        continuum.append(new)

    return continuum

continuum = make_continuum('metal' , 'wood')


In [21]:
# Make all the stim

materials = [s.material for s in samples]

for mat1 in materials:
    for mat2 in materials:
        if mat1 != mat2:
            cont = make_continuum(mat1, mat2, N=100)
            for i, s in enumerate(cont):
                for w in [40]:
                    sig = s.compile_signal() #what it was: (=200, decayDB=w)
                    scipy.io.wavfile.write('synth/'+ mat1 + '_' + mat2 + '_' + f"{i:02d}" + '-' + str(w) + '.wav',
                                          int(s.fs), sig)


In [22]:
# loop through the waves
# read in then new_sound = old_sound  + 16/32
# write the mp3 

#I'm gonna try loop through the .wav files that are on the computer - not sure if I can figure out how to loop through the waves directly. 

# Define the input and output directories
input_dir = "synth/"
output_dir = "synth/mp3Output/"


# Iterate over each file in the input directory

for file_name in os.listdir(input_dir):
    if file_name.endswith('.wav'):  # 
        # Load the audio file
        audio = AudioSegment.from_wav(os.path.join(input_dir, file_name))

        # Normalize the audio
        normalized_audio = audio + 1 #normalize(audio)

        # Export the normalized audio as MP3
        output_file_name = os.path.splitext(file_name)[0] + '.mp3'
        output_path = os.path.join(output_dir, output_file_name)
        normalized_audio.export(output_path, format="mp3")

        print(f"File '{file_name}' processed and saved as '{output_file_name}'")




File 'wood_glass_81-40.wav' processed and saved as 'wood_glass_81-40.mp3'
File 'wood_glass_78-40.wav' processed and saved as 'wood_glass_78-40.mp3'
File 'wood_glass_58-40.wav' processed and saved as 'wood_glass_58-40.mp3'
File 'wood_glass_82-40.wav' processed and saved as 'wood_glass_82-40.mp3'
File 'wood_glass_89-40.wav' processed and saved as 'wood_glass_89-40.mp3'
File 'wood_glass_91-40.wav' processed and saved as 'wood_glass_91-40.mp3'
File 'wood_glass_95-40.wav' processed and saved as 'wood_glass_95-40.mp3'
File 'wood_glass_51-40.wav' processed and saved as 'wood_glass_51-40.mp3'
File 'wood_glass_75-40.wav' processed and saved as 'wood_glass_75-40.mp3'
File 'wood_glass_32-40.wav' processed and saved as 'wood_glass_32-40.mp3'
File 'wood_glass_07-40.wav' processed and saved as 'wood_glass_07-40.mp3'
File 'wood_glass_74-40.wav' processed and saved as 'wood_glass_74-40.mp3'
File 'wood_glass_90-40.wav' processed and saved as 'wood_glass_90-40.mp3'
File 'wood_glass_29-40.wav' processed 

In [7]:
step = continuum[19]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [23]:
#this chunk creates variable declarations on the basis of the files we just created (everything in synth) 

#There's a few work arounds here because we have to change the way the names are structured so js can read the variables. 

# Set to store checked combinations to avoid duplicates
seen_combinations = set()

for file in os.listdir("synth/"):
    # Remove the file extension
    filename_without_extension = os.path.splitext(file)[0]
    # Split the filename into parts by '_'
    parts = filename_without_extension.split('_')
    
    non_numeric_parts = []
    numeric_parts = []
    
    for part in parts:
        # Handle parts that might be numeric or contain hyphens
        if part.isdigit():
            numeric_parts.append(part)
        elif '-' in part:
            # Split the part on '-' and check each segment for being numeric
            subparts = part.split('-')
            for subpart in subparts:
                if subpart.isdigit():
                    numeric_parts.append(subpart)
                else:
                    non_numeric_parts.append(subpart)
        else:
            non_numeric_parts.append(part)
    
    # Sort the non-numeric and numeric parts separately
    non_numeric_parts.sort()
    numeric_parts.sort(key=int)  # Sorting numerically
    
    # Create the standardized name by joining non-numeric parts first, then numeric
    standardized_name = "_".join(non_numeric_parts + numeric_parts)

    # Check if we've already seen this combination to avoid duplicates
    if standardized_name not in seen_combinations:
        seen_combinations.add(standardized_name)
        # Print the line of code to define the Howl object with all necessary formatting
        print(f"var {standardized_name} = new Howl({{\n  src: [\"/static/stimuli/audio/synth/{file}\"],\n  preload: true,\n}});")



var cardboard_metal_03_40 = new Howl({
  src: ["/static/stimuli/audio/synth/cardboard_metal_03-40.mp3"],
  preload: true,
});
var cardboard_metal_06_40 = new Howl({
  src: ["/static/stimuli/audio/synth/cardboard_metal_06-40.mp3"],
  preload: true,
});
var glass_wood_40_61 = new Howl({
  src: ["/static/stimuli/audio/synth/wood_glass_61-40.mp3"],
  preload: true,
});
var glass_wood_05_40 = new Howl({
  src: ["/static/stimuli/audio/synth/wood_glass_05-40.mp3"],
  preload: true,
});
var cardboard_metal_00_40 = new Howl({
  src: ["/static/stimuli/audio/synth/cardboard_metal_00-40.mp3"],
  preload: true,
});
var prevStims = new Howl({
  src: ["/static/stimuli/audio/synth/prevStims"],
  preload: true,
});
var cardboard_metal_01_40 = new Howl({
  src: ["/static/stimuli/audio/synth/cardboard_metal_01-40.mp3"],
  preload: true,
});
var glass_wood_00_40 = new Howl({
  src: ["/static/stimuli/audio/synth/wood_glass_00-40.mp3"],
  preload: true,
});
var cardboard_metal_08_40 = new Howl({
  src: ["/s

In [17]:
step = continuum[-1]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [3]:
#This chunk creates the array with unique identifiers based on the vars. (everything in synth, again).

# Set to store checked combinations to avoid duplicates
seen_combinations = set()
stims = []
unique_id = 1  # Starting unique identifier

for file in os.listdir("synth/"):
    # Remove the file extension
    basename = os.path.splitext(file)[0]
    parts = basename.split('_')
    
    non_numeric_parts = []
    numeric_parts = []
    
    for part in parts:
        if part.isdigit():
            numeric_parts.append(part)
        elif '-' in part:
            subparts = part.split('-')
            for subpart in subparts:
                if subpart.isdigit():
                    numeric_parts.append(subpart)
                else:
                    non_numeric_parts.append(subpart)
        else:
            non_numeric_parts.append(part)
    
    # Sort the non-numeric and numeric parts separately
    non_numeric_parts.sort()
    numeric_parts.sort(key=int)  # Sorting numerically
    
    # Create the standardized name by joining non-numeric parts first, then numeric
    standardized_name = "_".join(non_numeric_parts + numeric_parts)

    # Generate a category from the sorted non-numeric parts
    category = "_".join(non_numeric_parts)

    # Check if we've already seen this combination to avoid duplicates
    if standardized_name not in seen_combinations:
        seen_combinations.add(standardized_name)
        # Format unique_id as a three-digit number
        formatted_id = f"{unique_id:03}"
        stims.append([standardized_name, category, formatted_id])
        unique_id += 1  # Increment the unique identifier for the next entry

# Print the JavaScript variable
print("var stims = [")
for stim in stims:
    print(f'  ["{stim[0]}", "{stim[1]}", "{stim[2]}"],')
print("];")


var stims = [
  ["cardboard_metal_03_40", "cardboard_metal", "001"],
  ["ceramic_metal_06_40", "ceramic_metal", "002"],
  ["ceramic_metal_01_40", "ceramic_metal", "003"],
  ["ceramic_metal_04_40", "ceramic_metal", "004"],
  ["metal_wood_05_40", "metal_wood", "005"],
  ["ceramic_metal_07_40", "ceramic_metal", "006"],
  ["cardboard_wood_06_40", "cardboard_wood", "007"],
  ["cardboard_wood_04_40", "cardboard_wood", "008"],
  ["ceramic_metal_00_40", "ceramic_metal", "009"],
  ["ceramic_wood_09_40", "ceramic_wood", "010"],
  ["metal_wood_04_40", "metal_wood", "011"],
  ["cardboard_ceramic_00_40", "cardboard_ceramic", "012"],
  ["cardboard_ceramic_04_40", "cardboard_ceramic", "013"],
  ["metal_wood_07_40", "metal_wood", "014"],
  ["cardboard_metal_06_40", "cardboard_metal", "015"],
  ["ceramic_metal_03_40", "ceramic_metal", "016"],
  ["ceramic_wood_03_40", "ceramic_wood", "017"],
  ["ceramic_wood_07_40", "ceramic_wood", "018"],
  ["ceramic_wood_06_40", "ceramic_wood", "019"],
  ["ceramic_woo

In [23]:
step = continuum[-19]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [24]:
#this chunk creates the if-then function formatted the same way as the vars and array above. 

# Initialize the function string
function_str = """
var show_stim = function (sound) {
"""

# Directory containing the sound files
directory = "synth/"

# Set to store checked combinations to avoid duplicates
seen_combinations = set()

# Generate conditional statements for each sound file
for file in os.listdir(directory):
    basename = os.path.splitext(file)[0]  # Get the variable name without extension
    parts = basename.split('_')

    non_numeric_parts = []
    numeric_parts = []

    # Identify and separate numeric and non-numeric parts
    for part in parts:
        if part.isdigit():
            numeric_parts.append(part)
        elif '-' in part:
            # Handle hyphenated parts as potential numeric values
            subparts = part.split('-')
            for subpart in subparts:
                if subpart.isdigit():
                    numeric_parts.append(subpart)
                else:
                    non_numeric_parts.append(subpart)
        else:
            non_numeric_parts.append(part)

    # Sort the non-numeric and numeric parts
    non_numeric_parts.sort()
    numeric_parts.sort(key=int)  # Sort numerically

    # Create the standardized name
    standardized_name = "_".join(non_numeric_parts + numeric_parts)

    # Check if we've already added this combination
    if standardized_name not in seen_combinations:
        seen_combinations.add(standardized_name)
        function_str += f"    if (sound === '{standardized_name}') {{\n"
        function_str += f"      {standardized_name}.play();\n"
        function_str += "    } else "

# Remove the last 'else' and close the function
function_str = function_str.rstrip(' else ')
function_str += "{\n    }\n};\n"

print(function_str)



var show_stim = function (sound) {
    if (sound === 'cardboard_metal_03_40') {
      cardboard_metal_03_40.play();
    } else     if (sound === 'cardboard_metal_06_40') {
      cardboard_metal_06_40.play();
    } else     if (sound === 'glass_wood_40_61') {
      glass_wood_40_61.play();
    } else     if (sound === 'glass_wood_05_40') {
      glass_wood_05_40.play();
    } else     if (sound === 'cardboard_metal_00_40') {
      cardboard_metal_00_40.play();
    } else     if (sound === 'prevStims') {
      prevStims.play();
    } else     if (sound === 'cardboard_metal_01_40') {
      cardboard_metal_01_40.play();
    } else     if (sound === 'glass_wood_00_40') {
      glass_wood_00_40.play();
    } else     if (sound === 'cardboard_metal_08_40') {
      cardboard_metal_08_40.play();
    } else     if (sound === 'glass_wood_10_40') {
      glass_wood_10_40.play();
    } else     if (sound === 'glass_wood_40_81') {
      glass_wood_40_81.play();
    } else     if (sound === 'glass_w

In [12]:
#This chunk renames colours to be in order (otherwise their numbers are all over the place). 

import os

def rename_files_in_folder(folder_path):
    # List all files that match the pattern 'blue_green_*'
    files = sorted([file for file in os.listdir(folder_path) if file.startswith('blue_green_')])
    
    # Start numbering files from 001
    start_index = 1
    
    for index, file in enumerate(files, start=start_index):
        # Extract the file extension
        extension = os.path.splitext(file)[1]
        
        # Construct the new filename with the same prefix and new sequential number
        new_file_name = f"blue_green_{str(index).zfill(3)}{extension}"
        
        # Construct full old and new file paths
        old_file_path = os.path.join(folder_path, file)
        new_file_path = os.path.join(folder_path, new_file_name)
        # Rename file
        os.rename(old_file_path, new_file_path)
        print(f"Renamed '{file}' to '{new_file_name}'")

# Specify the folder path
folder_path = 'blue_green'
rename_files_in_folder(folder_path)



FileNotFoundError: [Errno 2] No such file or directory: 'blue_green'

In [13]:
#moving on to yellow-green folder - bit more complicated

import os

def sort_key(filename):
    # Extract the number from the filename and convert it to an integer for correct sorting
    parts = filename.split('_')
    return int(parts[-1].split('.')[0])

def rename_files_in_folder(folder_path):
    # List all files that match the pattern 'yellow_green_*'
    files = os.listdir(folder_path)
    # Filter and sort files by the numeric value
    files = sorted([file for file in files if file.startswith('red_green_')], key=sort_key)
    
    # Start numbering files from 001
    start_index = 1
    
    for index, file in enumerate(files, start=start_index):
        # Extract the file extension
        extension = os.path.splitext(file)[1]
        
        # Construct the new filename with the same prefix and new sequential number
        new_file_name = f"orange_yellow_{str(index).zfill(3)}{extension}"
        
        # Construct full old and new file paths
        old_file_path = os.path.join(folder_path, file)
        new_file_path = os.path.join(folder_path, new_file_name)
        
        # Rename file
        os.rename(old_file_path, new_file_path)
        print(f"Renamed '{file}' to '{new_file_name}'")

# Specify the folder path
folder_path = 'orange_yellow'
rename_files_in_folder(folder_path)



FileNotFoundError: [Errno 2] No such file or directory: 'orange_yellow'

In [11]:
import os
import xml.etree.ElementTree as ET
import re

def natural_sort_key(s):
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r'(\d+)', s)]

# Directory containing the SVG files
svg_directory = "allbats_copy"

# Initialize the list for the JavaScript array
stims = []

# Get a sorted list of SVG filenames using natural sorting (descending order)
svg_files = sorted([f for f in os.listdir(svg_directory) if f.endswith('.svg')], 
                  key=natural_sort_key, reverse=True)

# Loop through each SVG file in the directory in sorted order
for filename in svg_files:
    file_path = os.path.join(svg_directory, filename)
    try:
        # Parse the SVG file
        tree = ET.parse(file_path)
        root = tree.getroot()
        
        # Find the specific path with id="path3773"
        path3773 = root.find('.//*[@id="path3773"]')
        
        if path3773 is not None:
            # First try to get fill from direct attribute
            color = path3773.get('fill')
            
            # If not found in direct attribute, try to extract from style attribute
            if color is None:
                style = path3773.get('style')
                if style:
                    # Extract fill from style string (format: "fill:#HEX;...")
                    match = re.search(r'fill:(#[0-9a-fA-F]{3,6})', style)
                    if match:
                        color = match.group(1)
            
            # Remove the '.svg' extension from the filename
            basename = os.path.splitext(filename)[0]
            
            # Append formatted data to the stims array if color was found
            if color:
                stims.append(["stim", color, basename])
            else:
                print(f"Warning: No fill color found for path3773 in {filename}")
        else:
            print(f"Warning: path3773 not found in {filename}")
            
    except ET.ParseError as e:
        print(f"Error parsing {filename}: {str(e)}")
    except Exception as e:
        print(f"Unexpected error processing {filename}: {str(e)}")

# Print the JavaScript variable
print("var stims = [")
for stim in stims:
    print(f'  ["{stim[0]}", "{stim[1]}", "{stim[2]}"],')
print("];")

var stims = [
  ["stim", "#00cf62", "blue_green_99"],
  ["stim", "#00cf63", "blue_green_98"],
  ["stim", "#00ce65", "blue_green_97"],
  ["stim", "#00ce67", "blue_green_96"],
  ["stim", "#00ce69", "blue_green_95"],
  ["stim", "#00ce6a", "blue_green_94"],
  ["stim", "#00ce6c", "blue_green_93"],
  ["stim", "#00ce6e", "blue_green_92"],
  ["stim", "#00ce6f", "blue_green_91"],
  ["stim", "#00ce71", "blue_green_90"],
  ["stim", "#00ce73", "blue_green_89"],
  ["stim", "#00ce74", "blue_green_88"],
  ["stim", "#00ce76", "blue_green_87"],
  ["stim", "#00ce78", "blue_green_86"],
  ["stim", "#00ce79", "blue_green_85"],
  ["stim", "#00cd7b", "blue_green_84"],
  ["stim", "#00cd7d", "blue_green_83"],
  ["stim", "#00cd7e", "blue_green_82"],
  ["stim", "#00cd80", "blue_green_81"],
  ["stim", "#00cd82", "blue_green_80"],
  ["stim", "#00cd83", "blue_green_79"],
  ["stim", "#00cd85", "blue_green_78"],
  ["stim", "#00cd86", "blue_green_77"],
  ["stim", "#00cd88", "blue_green_76"],
  ["stim", "#00cd8a", "blu

In [103]:
step = continuum[-1]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [78]:
step = continuum[1]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [79]:
step = continuum[2]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [80]:
step = continuum[3]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [81]:
step = continuum[4]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [82]:
step = continuum[5]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [83]:
step = continuum[6]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [84]:
step = continuum[7]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [85]:
step = continuum[8]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [86]:
step = continuum[9]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [87]:
step = continuum[10]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [88]:
step = continuum[11]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [89]:
step = continuum[12]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [90]:
step = continuum[13]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [91]:
step = continuum[14]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [92]:
step = continuum[15]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [93]:
step = continuum[16]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [94]:
step = continuum[17]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [95]:
step = continuum[18]
sig = step.compile_signal()
Audio(sig, rate=step.fs)

In [98]:
step = continuum[19]
sig = step.compile_signal()
Audio(sig, rate=step.fs)